In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel

# 1. SETUP DATA
df = pd.read_csv('talabat_enhanced_orders.csv')
df['Order_Time'] = pd.to_datetime(df['Order_Time'])
df['order_hour'] = df['Order_Time'].dt.hour
df_fe = df.copy()

# Define common settings
target_col = 'Order_Status'
# Dropping leakage and ID columns
drop_cols = ['Order_ID', 'User_ID', 'Restaurant_ID', 'Driver_ID', 
             'Order_Time', 'Delivery_Time', 'Delivery_Duration_Minutes', 'Item_Name']

def get_accuracy(dataframe, top_k_val, peak_hours_list):
    """Helper to train and return accuracy and importances"""
    temp_df = dataframe.copy()
    
    # Task 3: Item Reduction
    top_items = temp_df["Item_Name"].value_counts().head(top_k_val).index
    temp_df["Item_Name_reduced"] = np.where(temp_df["Item_Name"].isin(top_items), temp_df["Item_Name"], "Other")
    
    # Task 2: Peak Hour Rule
    temp_df["is_peak_hour"] = temp_df["order_hour"].isin(peak_hours_list).astype(int)
    
    # Prepare X, y
    X = temp_df.drop(columns=drop_cols + [target_col])
    y = temp_df[target_col]
    
    cat_features = X.select_dtypes(include=['object']).columns.tolist()
    preprocess = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)], remainder='passthrough')
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    model = Pipeline([
        ('prep', preprocess),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced"))
    ])
    
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    
    # Get Feature Names and Importances
    ohe_names = model.named_steps['prep'].named_transformers_['cat'].get_feature_names_out()
    feature_names = list(ohe_names) + [col for col in X.columns if col not in cat_features]
    importances = model.named_steps['rf'].feature_importances_
    
    return acc, dict(zip(feature_names, importances))

# Baseline (Original Rules: k=20, peak=[12,13,14,15,19,20,21,22,23])
base_acc, base_imp = get_accuracy(df_fe, 20, [12,13,14,15,19,20,21,22,23])

In [29]:
# TASK 1: New Feature
df_fe["is_night_order"] = df_fe["order_hour"].apply(lambda x: 1 if x < 6 else 0)

In [30]:
# TASK 2: Modified Peak (Breakfast 7-9, Strict Lunch 13-14, Strict Dinner 20-22)
task2_acc, _ = get_accuracy(df_fe, 20, [7, 8, 9, 13, 14, 20, 21, 22])

In [ ]:
# TASK 3: Changed k (k=50)
task3_acc, task3_imp = get_accuracy(df_fe, 50, [7, 8, 9, 13, 14, 20, 21, 22])

print(f"Baseline Accuracy: {base_acc:.4f}")
print(f"Task 2 Accuracy (New Peak Rule): {task2_acc:.4f}")
print(f"Task 3 Accuracy (k=50): {task3_acc:.4f}")

print("\nTop 5 Feature Importances (k=50)")
sorted_imp = sorted(task3_imp.items(), key=lambda x: x[1], reverse=True)[:5]
for name, val in sorted_imp:
    print(f"{name}: {val:.4f}")

In [ ]:
# TASK 4: Feature Selection
top_items_50 = df_fe["Item_Name"].value_counts().head(50).index
df_fe["Item_Name_reduced"] = np.where(df_fe["Item_Name"].isin(top_items_50), df_fe["Item_Name"], "Other")
df_fe["is_peak_hour"] = df_fe["order_hour"].isin([7, 8, 9, 13, 14, 20, 21, 22]).astype(int)

X_task4 = df_fe.drop(columns=drop_cols + [target_col])
y_task4 = df_fe[target_col]
cat_f = X_task4.select_dtypes(include=['object']).columns.tolist()
prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat_f)], remainder='passthrough')

selector = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42), threshold="median")
model_fs = Pipeline([('prep', prep), ('select', selector), 
                     ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight="balanced"))])

X_train, X_test, y_train, y_test = train_test_split(X_task4, y_task4, test_size=0.2, random_state=42, stratify=y_task4)
model_fs.fit(X_train, y_train)
fs_acc = accuracy_score(y_test, model_fs.predict(X_test))

print(f"\nTask 4 Accuracy (Feature Selection): {fs_acc:.4f}")
print("\nTask 4 Report:")
print(classification_report(y_test, model_fs.predict(X_test), zero_division=0))